In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.applications.efficientnet import preprocess_input
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
train_real = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_train/face_real/*.jpg",shuffle=False)
train_fake = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_train/face_fake/*.jpg",shuffle=False)
test_real = tf.data.Dataset.list_files( "/content/drive/MyDrive/face_detection_test/face_real/*.jpg",shuffle=False)
test_fake = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_test/face_fake/*.jpg",shuffle=False)


In [3]:
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    return img
train_real = train_real.map(load_image)
train_fake = train_fake.map(load_image)
test_real = test_real.map(load_image)
test_fake = test_fake.map(load_image)

In [4]:
def add_label(image, label):
  return image , label

train_real = train_real.map(lambda x: add_label(x,0))
train_fake = train_fake.map(lambda x: add_label(x,1))
test_real = test_real.map(lambda x: add_label(x, 0))
test_fake = test_fake.map(lambda x: add_label(x, 1))

In [5]:
train_dataset = train_real.concatenate(train_fake)
test_dataset = test_real.concatenate(test_fake)
train_dataset = train_dataset.shuffle(7891)
#test_dataset = test_dataset.shuffle(2000)
train_size = int(0.8 * 7891)   # 6400
val_size = 8000 - train_size   # 1600

validation_dataset = train_dataset.skip(train_size)
train_dataset = train_dataset.take(train_size)

In [6]:
def preprocess(image, label):
    image = preprocess_input(image)
    return image, label
train_dataset = train_dataset.map(preprocess)
validation_dataset = validation_dataset.map(preprocess)
test_dataset = test_dataset.map(preprocess)


In [7]:
BATCH_SIZE = 32
train_dataset = train_dataset.batch(BATCH_SIZE)
validation_dataset = validation_dataset.batch(BATCH_SIZE)
test_dataset = test_dataset.batch(BATCH_SIZE)
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(AUTOTUNE)
validation_dataset = validation_dataset.prefetch(AUTOTUNE)
test_dataset = test_dataset.prefetch(AUTOTUNE)


In [19]:
model = tf.keras.models.load_model("/content/drive/MyDrive/efficientnet_stage2.keras")
base_model = model.layers[1]
base_model.trainable = True
for layers in base_model.layers[:-15]:
  layers.trainable = False


In [20]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
              loss="binary_crossentropy",
              metrics=["accuracy"])


In [21]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/efficientnet_finetuned4.keras",
    monitor="val_accuracy",
    save_best_only=True
)

history_finetune = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=15,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/15
198/198 ━━━━━━━━━━━━━━━━━━━━ 100s 272ms/step - accuracy: 0.6461 - loss: 0.6300 - val_accuracy: 0.7175 - val_loss: 0.5734
Epoch 2/15
198/198 ━━━━━━━━━━━━━━━━━━━━ 49s 148ms/step - accuracy: 0.7240 - loss: 0.5542 - val_accuracy: 0.8075 - val_loss: 0.4914
Epoch 3/15
198/198 ━━━━━━━━━━━━━━━━━━━━ 50s 150ms/step - accuracy: 0.7652 - loss: 0.5025 - val_accuracy: 0.8239 - val_loss: 0.4445
Epoch 4/15
198/198 ━━━━━━━━━━━━━━━━━━━━ 48s 145ms/step - accuracy: 0.7921 - loss: 0.4569 - val_accuracy: 0.8575 - val_loss: 0.3843
Epoch 5/15
198/198 ━━━━━━━━━━━━━━━━━━━━ 50s 148ms/step - accuracy: 0.8176 - loss: 0.4215 - val_accuracy: 0.8676 - val_loss: 0.3511
Epoch 6/15
198/198 ━━━━━━━━━━━━━━━━━━━━ 48s 142ms/step - accuracy: 0.8229 - loss: 0.4006 - val_accuracy: 0.8467 - val_loss: 0.3430
Epoch 7/15
198/198 ━━━━━━━━━━━━━━━━━━━━ 82s 148ms/step - accuracy: 0.8455 - loss: 0.3686 - val_accuracy: 0.8879 - val_loss: 0.3019
Epoch 8/15
198/198 ━━━━━━━━━━━━━━━━━━━━ 48s 144ms/step - accuracy: 0.8457 - loss: 

In [22]:
model.evaluate(test_dataset)

63/63 ━━━━━━━━━━━━━━━━━━━━ 9s 146ms/step - accuracy: 0.6430 - loss: 0.8039


[0.8038882613182068, 0.6430000066757202]